# Step 05 — Fine-tuning (QLoRA) Qwen2.5-3B-Instruct

Fine-tune the chosen base model to turn resume text into the project's `ResumeOutput` JSON, using 4-bit QLoRA so it fits a single 16 GB GPU.

What this notebook does:

1. Locates the project (or a Colab clone) and installs `requirements.txt`.
2. Downloads the curated `train`/`validation` splits and reviews them (schema validity, duplicates, token-length distribution).
3. Loads `Qwen/Qwen2.5-3B-Instruct` in 4-bit and attaches LoRA adapters.
4. Trains with **Weights & Biases** tracking and resumable checkpoints.
5. Scores the fine-tuned model on the validation split with the step-04 evaluator (directly comparable to the pre-fine-tuning baseline in `evals/`).
6. Pushes the adapter to the Hugging Face model repo.

Keys are read from **Google Colab's Secrets manager** (the key icon in the left sidebar) when running in Colab, and fall back to the project `.env` otherwise. Required: `HF_TOKEN`, `HF_MODEL_REPO`, `WANDB_API_KEY`; optional: `WANDB_ENTITY`, `WANDB_PROJECT`, `HF_DATASET_REPO`, `TYPESAFE_API_KEY`.

In [ ]:
# Fresh Colab runtimes start empty: clone the project, then work from inside it.
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Muhammad-Zain01/resume-parser-model.git'
CLONE_DIR = Path('/content/resume-parser-model')
MARKER = 'src/steps/05_fine_tuning/fine_tuning.py'


def _is_project(path):
    return (Path(path) / MARKER).is_file()


def _sync_repo(destination, url):
    destination = Path(destination)
    if (destination / '.git').is_dir():
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(destination)], check=True)


if _is_project(Path.cwd()):
    PROJECT_ROOT = Path.cwd().resolve()  # already inside the project (local run)
else:
    if not _is_project(CLONE_DIR):
        _sync_repo(CLONE_DIR, REPO_URL)
    PROJECT_ROOT = CLONE_DIR

os.chdir(PROJECT_ROOT)  # so requirements.txt and project-relative paths resolve
print('Project root:', PROJECT_ROOT)
print('Working directory:', Path.cwd())

In [ ]:
# Install the full dependency set (training stack: peft, trl, bitsandbytes, wandb, datasets).
%pip install -q -r {PROJECT_ROOT}/requirements.txt

In [ ]:
import json
import os
import sys
from importlib import import_module

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / '.env', override=False)
except ImportError:
    env_path = PROJECT_ROOT / '.env'
    if env_path.is_file():
        for line in env_path.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, _, value = line.partition('=')
                os.environ.setdefault(key.strip(), value.strip())

fine_tuning = import_module('src.steps.05_fine_tuning')
report_results = import_module('src.steps.04_evaluation').report_results
print('Fine-tuning module:', fine_tuning.__file__)

### Secrets -> environment variables

Pulls the project keys from **Google Colab's Secrets manager** (the key icon in the left sidebar) and exposes them as environment variables for every later cell. Colab values take precedence over `.env`. Outside Colab this is a no-op, so the `.env` file is used instead.

In [ ]:
SECRET_KEYS = [
    'HF_TOKEN',
    'HF_DATASET_REPO',
    'HF_DATASET_REVISION',
    'HF_MODEL_REPO',
    'HF_USERNAME',
    'WANDB_API_KEY',
    'WANDB_ENTITY',
    'WANDB_PROJECT',
    'TYPESAFE_API_KEY',
    'OPENROUTER_API_KEY',
]


def load_colab_secrets(keys=SECRET_KEYS):
    try:
        from google.colab import userdata  # available only inside Colab
    except ImportError:
        print('Not running in Colab; using .env / existing environment instead.')
        return {}
    loaded = {}
    for key in keys:
        try:
            value = userdata.get(key)
        except Exception:
            continue  # not defined or notebook access not granted
        if value:
            os.environ[key] = value  # Colab Secrets take precedence over .env
            loaded[key] = value
    return loaded


colab_secrets = load_colab_secrets()
print('Secrets loaded from Colab:', ', '.join(sorted(colab_secrets)) or 'none')

In [ ]:
# Confirm the keys this notebook needs are actually available before continuing.
REQUIRED_KEYS = ['HF_TOKEN', 'HF_MODEL_REPO', 'WANDB_API_KEY']
OPTIONAL_KEYS = ['WANDB_ENTITY', 'WANDB_PROJECT', 'HF_DATASET_REPO', 'TYPESAFE_API_KEY']


def _mask(value):
    if not value:
        return '(missing)'
    return value[:4] + '...' + value[-4:] if len(value) > 8 else '***'


print('Environment keys:')
for key in REQUIRED_KEYS + OPTIONAL_KEYS:
    value = os.getenv(key)
    if value:
        tag = 'OK'
    else:
        tag = 'MISSING (required)' if key in REQUIRED_KEYS else '(optional, unset)'
    print(f'  {key:20s} {tag:20s} {_mask(value)}')

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    raise RuntimeError(
        f"Missing required keys: {', '.join(missing)}. Add them to Colab Secrets (or .env)."
    )
print('All required keys are available.')

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
LOAD_IN_4BIT = True
MAX_SEQ_LENGTH = 8192

# Training hyperparameters (QLoRA).
EPOCHS = 2
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
SAVE_STEPS = 100
EVAL_STEPS = 100
MAX_TRAIN_SAMPLES = 2000  # set to None to train on the full 12,440-example split

# Validation scoring.
EVAL_SAMPLE = 20
MAX_NEW_TOKENS = 3072

# Weights & Biases experiment tracking.
USE_WANDB = True
WANDB_ENTITY = os.getenv('WANDB_ENTITY', 'zainmemon010-personal')
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'resume-parser')
RUN_NAME = 'qwen2.5-3b-resume-parser-qlora'

OUTPUT_DIR = fine_tuning.MODELS_DIR / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)
print('HF model repo:', fine_tuning.model_repo_id())
print('HF dataset repo:', fine_tuning.dataset_repo_id())

## 1. Download and verify the curated splits

Downloads only the curated `train`/`validation` files and reviews them before spending GPU time: schema validity, duplicates, and category coverage. Token-length stats are added after the tokenizer loads.

In [ ]:
splits = fine_tuning.download_curated_splits()
train_records = fine_tuning.load_records(splits['train'])
validation_records = fine_tuning.load_records(splits['validation'])
print(f'Train records: {len(train_records):,}')
print(f'Validation records: {len(validation_records):,}')

verification = fine_tuning.verify_dataset(train_records)
print(json.dumps({k: v for k, v in verification.items() if k != 'categories'}, indent=2))
print('Categories:', len(verification['categories']))

## 2. Load the base model in 4-bit (QLoRA)

Loads the tokenizer and the 4-bit quantized base model, then reports the prompt+completion token lengths so `MAX_SEQ_LENGTH` can be confirmed against the longest resumes.

In [ ]:
model, tokenizer = fine_tuning.load_model_and_tokenizer(MODEL_ID, load_in_4bit=LOAD_IN_4BIT)
print('Loaded', MODEL_ID, 'in 4-bit' if LOAD_IN_4BIT else 'in bf16')

lengths = fine_tuning.token_length_report(train_records, tokenizer, max_length=MAX_SEQ_LENGTH)
print('Token length report (train):')
print(json.dumps(lengths, indent=2))
if lengths['over_max_length']:
    print(f"WARNING: {lengths['over_max_length']} of {lengths['sampled']} sampled examples exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}.")

## 3. LoRA and training configuration

Builds the LoRA adapters, the `SFTConfig` (with periodic checkpoints), and the chat-format datasets. The user turn reuses the evaluation prompt so training matches evaluation.

In [ ]:
lora_config = fine_tuning.build_lora_config(r=16, alpha=32, dropout=0.05)
training_args = fine_tuning.build_training_arguments(
    OUTPUT_DIR,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_seq_length=MAX_SEQ_LENGTH,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    run_name=RUN_NAME,
    use_wandb=USE_WANDB,
)
train_dataset, eval_dataset = fine_tuning.build_datasets(
    train_records, validation_records, max_train=MAX_TRAIN_SAMPLES
)
print(train_dataset)
print(eval_dataset)

## 4. Train with Weights & Biases tracking

Training streams loss/LR and eval metrics to W&B and writes resumable checkpoints under `OUTPUT_DIR`. If Colab disconnects, restart, re-run from the top, and pass a checkpoint path to `resume_from_checkpoint` below.

In [ ]:
if USE_WANDB:
    fine_tuning.configure_wandb_env(
        entity=WANDB_ENTITY, project=WANDB_PROJECT, run_name=RUN_NAME
    )

trainer = fine_tuning.train(
    model,
    tokenizer,
    train_dataset,
    eval_dataset,
    lora_config=lora_config,
    training_args=training_args,
    resume_from_checkpoint=None,  # e.g. str(OUTPUT_DIR / 'checkpoint-500') to resume
)

In [ ]:
adapter_dir = fine_tuning.save_adapter(trainer, tokenizer, OUTPUT_DIR)
print('Saved adapter to:', adapter_dir)

## 5. Evaluate the fine-tuned model

Scores the tuned model on the validation split with the step-04 evaluator. Compare the resulting `mean_overall_score` and `overall_score_including_invalid_predictions` against the pre-fine-tuning baseline you saved in `evals/`.

In [ ]:
report = fine_tuning.evaluate_records(
    model,
    tokenizer,
    validation_records,
    model_name=f'finetuned:{MODEL_ID}',
    limit=EVAL_SAMPLE,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(json.dumps(report.metrics, indent=2))
summary = report_results({f'finetuned:{MODEL_ID}': report})

## 6. Push the adapter to the Hugging Face model repo

Uploads the LoRA adapter and tokenizer to `HF_MODEL_REPO`. The adapter is small (~tens of MB) and loads on top of the same 4-bit base.

In [ ]:
url = fine_tuning.push_adapter_to_hub(trainer.model, tokenizer, repo_id=fine_tuning.model_repo_id())
url

In [ ]:
# Optional: save the report files and release GPU memory before the next run.
# summary.save()
fine_tuning.free_memory()